In [1]:
# 1、读取日清分文件
# 2、获取日清分日期
# 3、获取场站名称
# 4、获取非滚撮中长期合约电量、电费
# 5、获取滚撮中长期合约电量、电费
# 6、获取日前结算量，计算日前中标量；获取实时结算量，获取上网电量（实时中标量），计算日前中标量，核对日前中标量
# 7、获取日前电价、实时电价，计算日前电费、实时电费，并核对
# 8、核对日前电费

In [6]:
import pandas as pd
import re
import numpy as np

# 设定文件路径（文件名中包含中文）
filename = 'D:/工作/特变电工/01政策与学习笔记/_全国_20250520/国网区域/新疆/400-交易复盘/现货日清分_系统导出数据/哈密市振超风力发电有限公司2025-04-27-明细.xlsx'

# 读取 Excel 文件（默认读取第一个 Sheet）
df = pd.read_excel(filename)
df.columns= ["名称"]+[f"{i}:00" for i in range(1, 25)]+["合计"]
# 读取场站名称
station_name=df.iloc[1,1]
print("电站名称是：",station_name)

# 用正则匹配“年-月-日”格式
match = re.search(r'\d{4}-\d{2}-\d{2}', filename)
if match:
    date_str = match.group()
    print("提取的日期是：", date_str)
else:
    print("未找到日期")
# 提取上网电量
generationData=df.iloc[3]

FileNotFoundError: [Errno 2] No such file or directory: 'D:/工作/特变电工/01政策与学习笔记/_全国_20250520/国网区域/新疆/400-交易复盘/现货日清分_系统导出数据/哈密市振超风力发电有限公司2025-04-27-明细.xlsx'

In [ ]:
## 合同序列数据封装
ContractDf=df.iloc[4:-8] # 获取df中的中长期合同序列数据
ContractList=[] # 构造列表容器，准备将合同装入
for i in range(0, len(ContractDf), 3):
    name = ContractDf.iloc[i,0]  # 从第 i 行第0列 提取合同序列名称
    quantity = ContractDf.iloc[i,1:] # 合同量
    price = ContractDf.iloc[i + 1,1:] # 合同价
    revenue = ContractDf.iloc[i + 2,1:] # 合同费用
    contract = ContractData(name, quantity, price, revenue) # 封装成对象
    ContractList.append(contract)   
# ContractList
## 市场分摊及返还费用数据封装
SpecialDf=df.iloc[-8:,:]
frRev=SpecialDf.iloc[0]
fr_allocation_Cost=SpecialDf.iloc[1]
unitPenaltyCostRecovery=SpecialDf.iloc[2]
unitPenaltyCostAllocation=SpecialDf.iloc[3]
GenMidLongRecovery=SpecialDf.iloc[4]
GenMidLongAllocation=SpecialDf.iloc[5]
ConsumerMidLongAllocation=SpecialDf.iloc[6]
ConsumerSpotAllocation=SpecialDf.iloc[7]
specialItemData=SpecialItemData(frRev, fr_allocation_Cost, unitPenaltyCostRecovery, unitPenaltyCostAllocation,
                 GenMidLongRecovery, GenMidLongAllocation,
                 ConsumerMidLongAllocation, ConsumerSpotAllocation)
## 场站日清分数据封装
stationDayData=StationDayData(station_name,date_str,generationData,ContractList,specialItemData)
stationDayData


In [17]:
#【df数据分块】
#【获取中长期合约MidLongContractDf】
MidLongContractDf=ContractDf.iloc[:-6]
# MidLongContractDf
#【获取现货SpotContractDf】
SpotContractDf=df.iloc[-14:-8,:]
# SpotContractDf
#【获取市场分摊与运营MarketAllocationOperationDf】
MarketAllocationOperationD=df.iloc[-8:]
# MarketAllocationOperationD

In [18]:
#【中长期合同量价费提取】
total_midLongQuantity_row,total_midLongRevenue_row,total_midLongPrice_row=QuantityRevenuePriceExtra(MidLongContractDf)
# print("中长期电量汇总：")
# print(total_midLongQuantity_row)
# print("\n中长期电费汇总：")
# print(total_midLongRevenue_row)
# print("\n中长期电价汇总：")
# print(total_midLongPrice_row)

In [19]:
# 【滚动撮合交易合同序列筛选】
# 找出包含“滚动撮合”的行索引
match_indicesRolling=MidLongContractDf[MidLongContractDf.iloc[:,0].str.contains("滚动撮合",na=False)].index
match_indicesRolling
# 初始化一个空列表来存储所有要保留的行索引
rollingRows_to_keep=[]
# 对每一个匹配的行，把它和它下面的两行都加入
for idx in match_indicesRolling:
    rollingRows_to_keep.extend([idx,idx+1,idx+2])
# 过滤出这些行
flitered_rollingContractDf=MidLongContractDf.loc[rollingRows_to_keep]
flitered_rollingContractDf

#【滚动撮合交易合同量价费提取】
total_rollingQuantity_row,total_rollingRevenue_row,total_rollingPrice_row=QuantityRevenuePriceExtra(flitered_rollingContractDf)
# print("滚动撮合交易电量汇总：")
# print(total_rollingQuantity_row)
# print("\n滚动撮合交易电费汇总：")
# print(total_rollingRevenue_row)
# print("\n滚动撮合交易电价汇总：")
# print(total_rollingPrice_row)

In [20]:
# 【获取非滚撮中长期合同序列】
midLongQuantity_row_withoutRolling=total_midLongQuantity_row-total_rollingQuantity_row
midLongRevenue_row_withoutRolling=total_midLongRevenue_row-total_rollingRevenue_row
midLongPrice_row_withoutRolling=midLongRevenue_row_withoutRolling/midLongQuantity_row_withoutRolling
# print("非滚动撮合中长期交易电量汇总：")
# print(midLongQuantity_row_withoutRolling)
# print("\n非滚动撮合中长期交易电费汇总：")
# print(midLongRevenue_row_withoutRolling)
# print("\n非滚动撮合中长期交易电价汇总：")
# print(midLongPrice_row_withoutRolling)

In [21]:
# 【现货日前交易量价费提取】
#现货日前结算电量(注意是结算不是出清----日前结算电量=日前出清电量-中长期合约电量)
dayAheadSettlementQuantity_row=SpotContractDf.iloc[0,:][1:]

#现货日前出清电价
dayAheadPrice_row=SpotContractDf.iloc[1,:][1:]
#现货日前结算电费
dayAheadSettlementRevenue_row=SpotContractDf.iloc[2,:][1:]
# 【现货实时交易量价费提取】
#现货实时结算电量(注意是结算不是出清----实时结算电量=实时出清电量-日前出清电量)
realtimeSettlementQuantity_row=SpotContractDf.iloc[3,:][1:]
#现货实时出清电价
realtimePrice_row=SpotContractDf.iloc[4,:][1:]
#现货实时结算电费
realtimeSettlementRevenue_row=SpotContractDf.iloc[5,:][1:]
#【现货日前、实时出清电量计算及校验】
# 日前出清电量=日前结算电量+中长期合约电量
# 日前出清电量=实时出清电量-实时结算电量
realtimeQuantity_row=generationData[1:]
dayahead1=dayAheadSettlementQuantity_row+total_midLongQuantity_row
dayahead2=realtimeQuantity_row-realtimeSettlementQuantity_row
if ((dayahead1 - dayahead2).abs().max() < 1e-4):
    dayAheadQuantity_row=dayahead1
    print(f"{station_name}{date_str}日清分结算单日前出清电量已生成。")
else:
    print(f"{station_name}{date_str}日清分结算单有误！请复核日前出清电量。")

# print(f"现货日前结算电量\n{dayAheadSettlementQuantity_row}")
# print(f"现货日前出清电量\n{dayAheadQuantity_row}")
# print(f"现货日前出清电价\n{dayAheadPrice_row}")
# print(f"现货日前结算电费\n{dayAheadSettlementRevenue_row}")
# print(f"现货实时结算电量\n{realtimeSettlementQuantity_row}")
# print(f"现货实时出清电量\n{realtimeQuantity_row}")
# print(f"现货实时出清电价\n{realtimePrice_row}")
# print(f"现货实时结算电费\n{realtimeSettlementRevenue_row}")

哈密市振超风力发电有限公司（振超分散式兴业风电一场）2025-04-27日清分结算单日前出清电量已生成。


In [22]:
# 【各时段实时-日前弃电情况】
# 实时-日前弃电率
curtailmentRate=(dayAheadQuantity_row-realtimeQuantity_row)/dayAheadQuantity_row
# 实时-日前弃电量
curtailmentQuantity=dayAheadQuantity_row-realtimeQuantity_row

# print(f"{station_name}{date_str}实时-日前弃电率：\n{curtailmentRate}")
# print(f"{station_name}{date_str}实时-日前弃电量：\n{curtailmentQuantity}")
# print(f"{station_name}{date_str}当日实时-日前平均弃电率为{float(curtailmentRate[24]):.2%}")

In [23]:
# 【生成dfQRP量费价Dataframe块】
# 【中长期dfQRP生成】
df_midLongQRP=seriesToMidLongDfQRP(total_midLongQuantity_row,total_midLongRevenue_row,total_midLongPrice_row,"中长期电量汇总","中长期电费汇总","中长期电价汇总")
# df_midLongQRP
df_midLongQRPwithoutRolling=seriesToMidLongDfQRP(midLongQuantity_row_withoutRolling,midLongRevenue_row_withoutRolling,midLongPrice_row_withoutRolling,"非滚动撮合中长期交易电量汇总","非滚动撮合中长期交易电费汇总","非滚动撮合中长期交易电价汇总")
# df_midLongQRPwithoutRolling
rollingQRP=seriesToMidLongDfQRP(total_rollingQuantity_row,total_rollingRevenue_row,total_rollingPrice_row,"滚动撮合交易电量汇总","滚动撮合交易电费汇总","滚动撮合交易电价汇总")
rollingQRP=rollingQRP.fillna(0)
# rollingQRP  #(注意此处滚撮QRP含NaN，运算时需要df.fillna(0)方式当成0处理)
# 【现货dfQRP生成】__现货QRP块多了一行现货结算电量
df_dayAheadQRP=seriesToSpotDfQRP(dayAheadSettlementQuantity_row,dayAheadQuantity_row,dayAheadSettlementRevenue_row,dayAheadPrice_row,
                                "现货日前结算电量","现货日前出清电量","现货日前结算电费","现货日前出清电价")
# df_dayAheadQRP
df_realtimeQRP=seriesToSpotDfQRP(realtimeSettlementQuantity_row,realtimeQuantity_row,realtimeSettlementRevenue_row,realtimePrice_row,
                                "现货实时结算电量","现货实时出清电量","现货实时结算电费","现货实时出清电价")
# df_realtimeQRP
##【生成市场运营及分摊df模块】
df_special=SpecialDf.reset_index(drop=True)
# df_special
## 【生成限电率df模块】
df_curtail=pd.concat([curtailmentRate.to_frame().T,curtailmentQuantity.to_frame().T],axis=0, ignore_index=True)
df_curtail.insert(loc=0, column="名称", value=["实时-日前弃电率","实时-日前弃电量"]) 
# df_curtail

In [24]:
# 【日清分账单分析】
#【日清分账单1】
df_dailySettlementType1 = pd.concat([df_midLongQRP,df_dayAheadQRP,df_realtimeQRP,df_special,df_curtail], ignore_index=True)
# print(df_dailySettlementType1)
#【日清分账单2】
df_dailySettlementType2=pd.concat([df_midLongQRPwithoutRolling,rollingQRP,df_dayAheadQRP,df_realtimeQRP,df_special,df_curtail], ignore_index=True)
# print(df_dailySettlementType2)


In [25]:
# 【日复盘type1-按照中长期所有合同汇总与现货进行分析】
#【日前-中长期复盘1】
# 日前出清电量-中长期合同电量汇总
df_dayAheadQuantityDifferType1=dayAheadQuantity_row.to_frame().T.reset_index(drop=True)-total_midLongQuantity_row.to_frame().T.reset_index(drop=True)
# print(df_dayAheadQuantityDifferType1)
# 日前出清电价-中长期合同电价汇总
df_dayAheadPriceDifferType1=dayAheadPrice_row.to_frame().T.reset_index(drop=True)-total_midLongPrice_row.to_frame().T.reset_index(drop=True)
# df_dayAheadPriceDifferType1
# 日前-中长期盈亏情况
df_dayAheadProfitAndLossValueType1=df_dayAheadQuantityDifferType1*df_dayAheadPriceDifferType1
# df_dayAheadProfitAndLossValueType1
df_dayAheadProfitAndLossConditionType1=dayAheadProfitLossCondition(df_dayAheadQuantityDifferType1,df_dayAheadPriceDifferType1)
# df_dayAheadProfitAndLossConditionType1
df_dayAheadQuantityDifferType1.insert(loc=0, column="名称", value=["日前中长期量差"]) 
df_dayAheadPriceDifferType1.insert(loc=0, column="名称", value=["日前中长期价差"]) 
df_dayAheadProfitAndLossValueType1.insert(loc=0, column="名称", value=["日前中长期盈亏值"]) 
df_dayAheadProfitAndLossConditionType1.insert(loc=0, column="名称", value=["日前中长期盈亏状态"]) 
# 日前复盘df块（type1）
df_dayAheadReviewType1=pd.concat([df_dayAheadQuantityDifferType1,df_dayAheadPriceDifferType1,df_dayAheadProfitAndLossValueType1,df_dayAheadProfitAndLossConditionType1],ignore_index=True)
# print(df_dayAheadReviewType1)
# 【实时-日前复盘1】
# 实时出清电量-日前出清电量
df_realtimeQuantityDifferType1=realtimeQuantity_row.to_frame().T.reset_index(drop=True)-dayAheadQuantity_row.to_frame().T.reset_index(drop=True)
# df_realtimeQuantityDifferType1
# 实时出清电价-日前出清电价
df_realtimePriceDifferType1=realtimePrice_row.to_frame().T.reset_index(drop=True)-dayAheadPrice_row.to_frame().T.reset_index(drop=True)
# df_realtimePriceDifferType1
# 实时-日前盈亏情况
df_realtimeProfitAndLossValueType1=df_realtimeQuantityDifferType1*df_realtimePriceDifferType1
# df_realtimeProfitAndLossValueType1
df_realtimeProfitAndLossConditionType1=realtimerofitLossCondition(df_realtimeQuantityDifferType1,df_realtimePriceDifferType1)
# df_realtimeProfitAndLossConditionType1
df_realtimeQuantityDifferType1.insert(loc=0, column="名称", value=["实时日前量差"]) 
df_realtimePriceDifferType1.insert(loc=0, column="名称", value=["实时日前价差"]) 
df_realtimeProfitAndLossValueType1.insert(loc=0, column="名称", value=["实时日前盈亏值"]) 
df_realtimeProfitAndLossConditionType1.insert(loc=0, column="名称", value=["实时日前盈亏状态"]) 
# 实时复盘df块（type1）
df_realtimeReviewType1=pd.concat([df_realtimeQuantityDifferType1,df_realtimePriceDifferType1,df_realtimeProfitAndLossValueType1,df_realtimeProfitAndLossConditionType1],ignore_index=True)
# print(df_realtimeReviewType1)
# 日复盘df块（type1）
df_dailyReviewType1=pd.concat([df_midLongQRP,df_dayAheadQRP,df_realtimeQRP,df_special,df_curtail, df_dayAheadReviewType1,df_realtimeReviewType1], ignore_index=True)
# df_dailyReviewType1

In [26]:
# 【日复盘type2-按照滚撮合同与现货进行分析】
# 【滚撮-非滚撮中长期复盘】
# 滚撮-非滚撮中长期合同电量
df_rollingQuantityDiffer=total_rollingQuantity_row.to_frame().T.fillna(0).reset_index(drop=True)-midLongQuantity_row_withoutRolling.to_frame().T.fillna(0).reset_index(drop=True)
# df_RollingQuantityDiffer
# 滚撮-非滚撮中长期合同电价
df_rollingPriceDiffer=total_rollingPrice_row.to_frame().T.fillna(0).reset_index(drop=True)-midLongPrice_row_withoutRolling.to_frame().T.fillna(0).reset_index(drop=True)
# df_RollingPriceDiffer
# 滚撮-非滚撮中长期盈亏情况
df_rollingProfitAndLossValue=df_rollingQuantityDiffer*df_rollingPriceDiffer
# df_rollingProfitAndLossValue
df_rollingProfitAndLossCondition=rollingProfitLossCondition(df_rollingQuantityDiffer,df_rollingPriceDiffer)
# df_rollingProfitAndLossCondition
df_rollingQuantityDiffer.insert(loc=0, column="名称", value=["滚撮中长期量差"]) 
df_rollingPriceDiffer.insert(loc=0, column="名称", value=["滚撮中长期价差"]) 
df_rollingProfitAndLossValue.insert(loc=0, column="名称", value=["滚撮中长期盈亏值"]) 
df_rollingProfitAndLossCondition.insert(loc=0, column="名称", value=["滚撮中长期盈亏状态"]) 
# print(df_rollingQuantityDiffer)
# print(df_rollingPriceDiffer)
# print(df_rollingProfitAndLossValue)
# print(df_rollingProfitAndLossCondition)
# 滚撮复盘df块（type2）
df_rollingReview=pd.concat([df_rollingQuantityDiffer,df_rollingPriceDiffer,df_rollingProfitAndLossValue,df_rollingProfitAndLossCondition],ignore_index=True)
# print(df_rollingReview)
#【日前-滚撮复盘】
# 日前出清电量-滚撮合同电量
df_dayAheadQuantityDifferType2=dayAheadQuantity_row.to_frame().T.reset_index(drop=True)-total_rollingQuantity_row.to_frame().T.fillna(0).reset_index(drop=True)
# df_dayAheadQuantityDifferType2
# 日前出清电价-滚撮合同电价
df_dayAheadPriceDifferType2=dayAheadPrice_row.to_frame().T.reset_index(drop=True)-total_rollingPrice_row.to_frame().T.fillna(0).reset_index(drop=True)
df_dayAheadPriceDifferType2
# 日前-滚撮盈亏情况
df_dayAheadProfitAndLossValueType2=df_dayAheadQuantityDifferType2*df_dayAheadPriceDifferType2
df_dayAheadProfitAndLossValueType2
df_dayAheadProfitAndLossConditionType2=dayAheadProfitLossCondition(df_dayAheadQuantityDifferType2,df_dayAheadPriceDifferType2)
df_dayAheadProfitAndLossConditionType2
df_dayAheadQuantityDifferType2.insert(loc=0, column="名称", value=["日前滚撮量差"]) 
df_dayAheadPriceDifferType2.insert(loc=0, column="名称", value=["日前滚撮价差"]) 
df_dayAheadProfitAndLossValueType2.insert(loc=0, column="名称", value=["日前滚撮盈亏值"]) 
df_dayAheadProfitAndLossConditionType2.insert(loc=0, column="名称", value=["日前滚撮盈亏状态"]) 
# 日前复盘df块（type2）
df_dayAheadReviewType2=pd.concat([df_dayAheadQuantityDifferType2,df_dayAheadPriceDifferType2,df_dayAheadProfitAndLossValueType2,df_dayAheadProfitAndLossConditionType2],ignore_index=True)
# print(df_dayAheadReviewType2)
# 【实时-日前复盘2】
# 实时出清电量-日前出清电量
df_realtimeQuantityDifferType2=realtimeQuantity_row.to_frame().T.reset_index(drop=True)-dayAheadQuantity_row.to_frame().T.reset_index(drop=True)
# df_realtimeQuantityDifferType2
# 实时出清电价-日前出清电价
df_realtimePriceDifferType2=realtimePrice_row.to_frame().T.reset_index(drop=True)-dayAheadPrice_row.to_frame().T.reset_index(drop=True)
# df_realtimePriceDifferType2
# 实时-日前盈亏情况
df_realtimeProfitAndLossValueType2=df_realtimeQuantityDifferType2*df_realtimePriceDifferType2
# df_realtimeProfitAndLossValueType1
df_realtimeProfitAndLossConditionType2=realtimerofitLossCondition(df_realtimeQuantityDifferType2,df_realtimePriceDifferType2)
# df_realtimeProfitAndLossConditionType2
df_realtimeQuantityDifferType2.insert(loc=0, column="名称", value=["实时日前量差"]) 
df_realtimePriceDifferType2.insert(loc=0, column="名称", value=["实时日前价差"]) 
df_realtimeProfitAndLossValueType2.insert(loc=0, column="名称", value=["实时日前盈亏值"]) 
df_realtimeProfitAndLossConditionType2.insert(loc=0, column="名称", value=["实时日前盈亏状态"]) 
# 实时复盘df块（type2）
df_realtimeReviewType2=pd.concat([df_realtimeQuantityDifferType2,df_realtimePriceDifferType2,df_realtimeProfitAndLossValueType2,df_realtimeProfitAndLossConditionType2],ignore_index=True)
# print(df_realtimeReviewType2)
# 日复盘df块（type2）
df_dailyReviewType2=pd.concat([df_midLongQRPwithoutRolling,rollingQRP,df_dayAheadQRP,df_realtimeQRP,df_special,df_curtail,df_rollingReview, df_dayAheadReviewType2,df_realtimeReviewType2], ignore_index=True)
# df_dailyReviewType2



In [27]:

# 设置输出文件路径
out_file_path = f'D:/工作/特变电工/01政策与学习笔记/_全国_20250520/国网区域/新疆/400-交易复盘/现货日复盘/{date_str}_{station_name}_现货日清分复盘.xlsx'

# 使用 pd.ExcelWriter 将多个 DataFrame 写入不同 sheet
with pd.ExcelWriter(out_file_path, engine='openpyxl') as writer:
    df_dailySettlementType1.to_excel(writer, sheet_name='日清分单整理Type1', index=False)
    df_dailySettlementType2.to_excel(writer, sheet_name='日清分单整理Type2', index=False)
    df_dailyReviewType1.to_excel(writer, sheet_name='日复盘Type1', index=False)
    df_dailyReviewType2.to_excel(writer, sheet_name='日复盘Type2', index=False)



In [1]:
# 【类定义】
# 合同序列类
class ContractData:
    def __init__(self, name, quantity, price, revenue):
        self.name = name            # 合同名称
        self.quantity = quantity    # 第一行，24小时电量
        self.price = price          # 第二行，24小时价格
        self.revenue = revenue            # 第三行，24小时费用

    def total_quantity(self):
        return self.quantity[:24].sum()
    
    def total_revenue(self):
        return self.revenue[:24].sum()
    
    def avg_price(self):
        return (self.price[:24] * self.quantity[:24]).sum() / self.quantity[:24].sum()
    def __repr__(self):
        return (
            f"<交易合同序列={self.name} "
            f"合同总量={self.total_quantity():.2f} "
            f"加权均价={self.avg_price():.2f} "
            f"总费用={self.total_revenue():.2f}>"
        )    

# 市场分摊及返还费用类
class SpecialItemData:
    def __init__(self, frRev, fr_allocation_Cost, unitPenaltyCostRecovery, unitPenaltyCostAllocation,
                 GenMidLongRecovery, GenMidLongAllocation,
                 ConsumerMidLongAllocation, ConsumerSpotAllocation):
        self.frRev = frRev   # 调频辅助服务里程收益 
        self.fr_allocation_Cost=fr_allocation_Cost # 调频辅助服务费用分摊
        self.unitPenaltyCostRecovery=unitPenaltyCostRecovery # 机组考核费用-回收
        self.unitPenaltyCostAllocation=unitPenaltyCostAllocation # 机组考核费用-分摊
        self.GenMidLongRecovery=GenMidLongRecovery # 发电侧中长期偏差收益回收-回收
        self.GenMidLongAllocation=GenMidLongAllocation # 发电侧中长期偏差收益回收-分摊
        self.ConsumerMidLongAllocation=ConsumerMidLongAllocation # 用户侧中长期偏差收益回收-分摊
        self.ConsumerSpotAllocation=ConsumerSpotAllocation # 用户侧现货偏差收益回收-分摊
    def __repr__(self):
        return (
            f"<调频辅助服务里程收益={self.frRev[-1]} "
            f"调频辅助服务费用分摊={self.fr_allocation_Cost[-1]} "
            f"机组考核费用-回收={self.unitPenaltyCostRecovery[-1]} "
            f"机组考核费用-分摊={self.unitPenaltyCostAllocation[-1]} "
            f"发电侧中长期偏差收益回收-回收={self.GenMidLongRecovery[-1]} "
            f"发电侧中长期偏差收益回收-分摊={self.GenMidLongAllocation[-1]} "
            f"用户侧中长期偏差收益回收-分摊={self.ConsumerMidLongAllocation[-1]} "
            f"用户侧现货偏差收益回收-分摊={self.ConsumerSpotAllocation[-1]}>"
        )      

# 定义场站日清分数据类 
class StationDayData:
    def __init__(self, station_name, date,generation, contracts,specail_item):
        self.station_name = station_name # 场站名称
        self.date = date  # 清分日期
        self.generation=generation # 上网数据
        self.contracts = contracts  # List of ContractData
        self.specail_item = specail_item # 市场分摊及返还费用
    def __repr__(self):
        return(
            f"<场站日清分数据\n"
            f"场站名称：{self.station_name}"
            f"清分日期：{self.date}>"
        )
    
# 定义所有场站日清分汇总数据类
class PowerDataBook:
    def __init__(self):
        # data 是一个嵌套字典：{station_name: {date: StationDayData}}
        self.data = {}  # dict[station_name][date] = StationDayData
     
    def add_station_day_data(self, station_day_data):
        """添加一条站点的某天数据"""
        s = station_day_data.station_name # 场站名称
        d = station_day_data.date # 日期
        if s not in self.data:
            self.data[s] = {} # 不包含场站时，创建新场站
        self.data[s][d] = station_day_data #添加场站s在d日的日清分数据

In [2]:
# 【定义日清分合同汇总类】
class ContractSummaryData:
    def __init__(self, station_name, date,generation, QmidLong,RmidLong,Qrolling,Rrolling,Qdayahead,Pdayahead,Rdayahead,Qrealtime,Prealtime,Rrealtime):
        self.station_name = station_name # 场站名称
        self.date = date  # 清分日期
        self.generation=generation
        self.QmidLong=QmidLong
        self.RmidLong=RmidLong
        self.Qrolling=Qrolling
        self.Rrolling=Rrolling
        self.Qdayahead=Qdayahead
        self.Pdayahead=Pdayahead
        self.Rdayahead=Rdayahead
        self.Qrealtime=Qrealtime
        self.Prealtime=Prealtime
        self.Rrealtime=Rrealtime

In [3]:
#【量费价汇总提取模块】
def QuantityRevenuePriceExtra(DF):
    total_Quantity_row=pd.Series(0,index=DF.columns)
    total_Revenue_row=pd.Series(0,index=DF.columns)
    total_Quantity_row=total_Quantity_row[1:]
    total_Revenue_row=total_Revenue_row[1:]
    # 每隔3行是一个品种的数据：第0行电量、第2行电费
    for i in range(0, len(DF), 3):
        total_Quantity_row += DF.iloc[i,1:]
        total_Revenue_row += DF.iloc[i + 2,1:]
    # 使用 np.divide 安全相除，避免 ZeroDivisionError
    # 使用 np.divide + where 条件，确保不除以 0
    total_Price_row = pd.Series(
        np.divide(
            total_Revenue_row.values.astype(float),
            total_Quantity_row.values.astype(float),
            out=np.full_like(total_Revenue_row.values, np.nan, dtype='float64'),
            where=total_Quantity_row.values.astype(float) != 0
        ),
        index=total_Revenue_row.index
    ) 
    # 返回提取的量、费、价
    return total_Quantity_row, total_Revenue_row, total_Price_row

In [4]:
#【中长期series转df量费价QRP模块】
def seriesToMidLongDfQRP(seriesQuantityRow,seriesRevenueRow,seriesPriceRow,rowNameQuantity,rowNameRevenue,rowNamePrice):
    """
    参数：
    seriesQuantityRow:合同电量Series
    seriesRevenueRow:合同电费Series
    seriesPriceRow:合同电价Series
    rowNameQuantity:合同电量行名string
    rowNameRevenue:合同电费行名string
    rowNamePrice:合同电价行名string
    返回：
    dfQRP:量费价df块
    
    """
    dfQRP=pd.concat([seriesQuantityRow.to_frame().T, seriesRevenueRow.to_frame().T,seriesPriceRow.to_frame().T],axis=0, ignore_index=True)
    dfQRP.insert(loc=0, column="名称", value=[rowNameQuantity,rowNameRevenue,rowNamePrice]) 
    return dfQRP
#【现货series转df量费价QRP模块】
def seriesToSpotDfQRP(settlementQuantityRow,seriesQuantityRow,seriesRevenueRow,seriesPriceRow,rowNameSettlementQuantity,rowNameQuantity,rowNameRevenue,rowNamePrice):
    """
    参数：
    settlementQuantityRow:结算电量Series
    seriesQuantityRow:现货出清电量Series
    seriesRevenueRow:合同电费Series
    seriesPriceRow:合同电价Series
    rowNameSettlementQuantity:结算电量行名string
    rowNameQuantity:现货出清电量行名string
    rowNameRevenue:合同电费行名string
    rowNamePrice:合同电价行名string
    返回：
    dfQRP:量费价df块
    
    """
    dfQRP=pd.concat([settlementQuantityRow.to_frame().T,seriesQuantityRow.to_frame().T, seriesRevenueRow.to_frame().T,seriesPriceRow.to_frame().T],axis=0, ignore_index=True)
    dfQRP.insert(loc=0, column="名称", value=[rowNameSettlementQuantity,rowNameQuantity,rowNameRevenue,rowNamePrice]) 
    return dfQRP

In [5]:
# 【滚撮中长期套利盈亏状态分析模块】
def rollingProfitLossCondition(df_quantityDiffer,df_priceDiffer):
    # 构建条件判断
    cond_a = (df_quantityDiffer >= 0) & (df_priceDiffer >= 0) # 套利（滚撮高价卖出）
    cond_b = (df_quantityDiffer > 0) & (df_priceDiffer< 0) #亏损（滚撮低价卖出）
    cond_c = (df_quantityDiffer < 0) & (df_priceDiffer > 0) #亏损（滚撮高价反购）
    cond_d = (df_quantityDiffer <= 0) & (df_priceDiffer <= 0) #套利（滚撮低价反购）
    # 用 numpy 的 select 创建新 DataFrame
    choices = ['套利（滚撮高价卖出）', '亏损（滚撮低价卖出）', '亏损（滚撮高价反购）', '套利（滚撮低价反购）']
    conditions = [cond_a, cond_b, cond_c, cond_d]
    df_Condition = pd.DataFrame(
        np.select(conditions, choices, default=''),
        columns=df_quantityDiffer.columns
    )
    return df_Condition



# 【日前中长期套利盈亏状态分析模块】
def dayAheadProfitLossCondition(df_quantityDiffer,df_priceDiffer):
    # 构建条件判断
    cond_a = (df_quantityDiffer >= 0) & (df_priceDiffer >= 0) # 套利（日前高价卖出）
    cond_b = (df_quantityDiffer > 0) & (df_priceDiffer< 0) #亏损（日前低价卖出）
    cond_c = (df_quantityDiffer < 0) & (df_priceDiffer > 0) #亏损（日前高价反购）
    cond_d = (df_quantityDiffer <= 0) & (df_priceDiffer <= 0) #套利（日前低价反购）
    # 用 numpy 的 select 创建新 DataFrame
    choices = ['套利（日前高价卖出）', '亏损（日前低价卖出）', '亏损（日前高价反购）', '套利（日前低价反购）']
    conditions = [cond_a, cond_b, cond_c, cond_d]
    df_Condition = pd.DataFrame(
        np.select(conditions, choices, default=''),
        columns=df_quantityDiffer.columns
    )
    return df_Condition
# 【实时日前套利盈亏状态分析模块】
def realtimerofitLossCondition(df_quantityDiffer,df_priceDiffer):
    # 构建条件判断
    cond_a = (df_quantityDiffer >= 0) & (df_priceDiffer >= 0) # 套利（实时高价卖出）
    cond_b = (df_quantityDiffer > 0) & (df_priceDiffer< 0) #亏损（实时低价卖出）
    cond_c = (df_quantityDiffer < 0) & (df_priceDiffer > 0) #亏损（实时高价反购）
    cond_d = (df_quantityDiffer <= 0) & (df_priceDiffer <= 0) #套利（实时低价反购）
    # 用 numpy 的 select 创建新 DataFrame
    choices = ['套利（实时高价卖出）', '亏损（实时低价卖出）', '亏损（实时高价反购）', '套利（实时低价反购）']
    conditions = [cond_a, cond_b, cond_c, cond_d]
    df_Condition = pd.DataFrame(
        np.select(conditions, choices, default=''),
        columns=df_quantityDiffer.columns
    )
    return df_Condition


